# deep_taste — BPR training (GPU)

Trains the two-tower recommender on a Colab GPU. Much faster than local MPS, and
CUDA sidesteps the MPS-specific numerical quirks seen during debugging.

**Runtime → Change runtime type → T4 GPU** before running.

**Upload to Google Drive in a folder named `deep_taste`:**
- `data/processed/features.pt` (40 MB)
- `data/processed/reviews_split.parquet` (179 MB)
- `src/model.py`
- `src/train.py`

In [ ]:
# 1. Confirm GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Deps (torch preinstalled on Colab; train.py needs pandas/pyarrow which are too)
!pip install -q pyarrow

In [ ]:
# 3. Mount Drive and stage files onto local disk
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/deep_taste'
!mkdir -p /content/data/processed /content/src
!cp $DRIVE/features.pt $DRIVE/reviews_split.parquet /content/data/processed/
!cp $DRIVE/model.py $DRIVE/train.py /content/src/
!ls -la /content/data/processed/

In [ ]:
# 4. Train. train.py resolves paths relative to CWD, so run from /content.
#    Safe defaults after the dying-ReLU collapse: lr 3e-4, grad clip 1.0.
%cd /content
!python src/train.py --epochs 20 --batch-size 2048 --lr 3e-4 --clip 1.0

In [ ]:
# 5. Sanity-check the trained encoder before copying back: are embeddings diverse
#    (not collapsed) and do positives outscore a random restaurant?
import sys; sys.path.insert(0, '/content/src')
import torch
from model import RestaurantEncoder
feats = torch.load('/content/data/processed/features.pt', weights_only=False)
enc = RestaurantEncoder(feats, output_dims=128)
enc.load_state_dict(torch.load('/content/data/processed/encoder.pt')); enc.eval()
with torch.no_grad():
    emb = enc(torch.arange(len(feats['business_ids'])))
print('mean pairwise cosine (500):', float((emb[:500] @ emb[:500].T).mean()))
print('per-dim std:', float(emb.std(0).mean()), '  (near 0 => collapsed)')

In [ ]:
# 6. Copy the trained weights back to Drive
!cp /content/data/processed/encoder.pt $DRIVE/
!ls -lah $DRIVE/encoder.pt